# 🎯 Modelos de Clasificación Supervisada y Evaluación de Impacto de Negocio

### Autor: Néstor León | Business Intelligence & Analytics
---

En un entorno comercial real, predecir si un cliente va a contratar o cancelar un producto no es suficiente si no entendemos **por qué toma esa decisión** y **cómo priorizar las acciones operativas**.

En este notebook:
1. Entrenamos y comparamos cuatro modelos de clasificación representativos:
   - **Regresión Logística**: Modelo de referencia probabilístico con cálculo de *Odds Ratios*.
   - **Árbol de Decisión (CART)**: Totalmente interpretable con reglas visuales claras.
   - **Gaussian Naive Bayes**: Clasificador probabilístico rápido basado en el Teorema de Bayes.
   - **Random Forest**: Ensamble robusto para capturar no-linealidades complejas.
2. Evaluamos los modelos con métricas que importan al negocio: Curva ROC-AUC, Precision, Recall y **Curva Lift de Ganancia Acumulada**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

print("Módulos de clasificación importados.")


## 1. Preparación de Datos y Partición Estratificada

Trabajamos con el problema de predecir si un cliente contrata el producto `pep` (1 = Sí, 0 = No). Aplicamos partición estratificada 80/20 para conservar la proporción del target en train y test.


In [ ]:
# Generamos dataset con relaciones reales de propensión
np.random.seed(42)
N = 1000

df_bank = pd.DataFrame({
    'edad': np.random.randint(18, 68, N),
    'ingresos': np.random.uniform(15000, 75000, N),
    'hijos': np.random.choice([0, 1, 2, 3], N, p=[0.4, 0.3, 0.2, 0.1]),
    'ahorro_activo': np.random.choice([0, 1], N, p=[0.3, 0.7]),
    'hipoteca': np.random.choice([0, 1], N, p=[0.6, 0.4])
})

# Probabilidad de contratar condicionada por variables
logit = -2.5 + 0.03 * df_bank['edad'] + 0.00004 * df_bank['ingresos'] + 0.5 * df_bank['ahorro_activo'] - 0.4 * df_bank['hipoteca']
prob = 1 / (1 + np.exp(-logit))
df_bank['contrata_producto'] = (prob > np.random.uniform(0, 1, N)).astype(int)

X = df_bank.drop('contrata_producto', axis=1)
y = df_bank['contrata_producto']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Tamaño Train: {len(X_train)} | Tamaño Test: {len(X_test)}")
print(f"Tasa de conversión global: {y.mean() * 100:.1f}%")


## 2. Entrenamiento y Comparativa de Modelos

Evaluamos los cuatro modelos mediante validación cruzada de 5 particiones estratificadas (*Stratified 5-Fold CV*) midiendo el área bajo la curva ROC (ROC-AUC).


In [ ]:
modelos = {
    'Regresión Logística': LogisticRegression(max_iter=1000),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados_cv = {}

for nombre, mod in modelos.items():
    scores = cross_val_score(mod, X_train, y_train, cv=cv, scoring='roc_auc')
    resultados_cv[nombre] = scores
    print(f"{nombre}: ROC-AUC medio = {scores.mean():.4f} (+/- {scores.std():.4f})")


## 3. Interpretabilidad: Regresión Logística y Odds Ratios

La dirección de negocio siempre pregunta: *¿Cuánto más probable es que un cliente contrate el producto si tiene una cuenta de ahorro activa?*

Los **Odds Ratios** ($e^eta$) dan la respuesta directa en términos de probabilidad relativa.


In [ ]:
log_reg = LogisticRegression(max_iter=1000).fit(X_train, y_train)

df_odds = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente_Beta': log_reg.coef_[0],
    'Odds_Ratio': np.exp(log_reg.coef_[0])
}).sort_values(by='Odds_Ratio', ascending=False)

df_odds


## 4. Comparativa de Curvas ROC en el Conjunto de Test

Ajustamos todos los modelos sobre el conjunto de test para evaluar su capacidad de discriminación fuera de muestra.


In [ ]:
plt.figure(figsize=(9, 6))

for nombre, mod in modelos.items():
    mod.fit(X_train, y_train)
    y_prob = mod.predict_proba(X_test)[:, 1]
    auc_val = roc_auc_score(y_test, y_prob)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{nombre} (AUC = {auc_val:.3f})", lw=2)

plt.plot([0, 1], [0, 1], 'k--', label='Aleatorio (AUC = 0.50)')
plt.title("Curvas ROC Comparativas en Conjunto de Test", fontsize=14, fontweight='bold')
plt.xlabel("Tasa de Falsos Positivos (FPR)")
plt.ylabel("Tasa de Verdaderos Positivos (TPR / Recall)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


## 5. Análisis de la Curva Lift y Ganancia Acumulada

En una campaña de telemarketing, llamar al 100% de la base es costoso e ineficiente. La **Curva Lift** nos indica cuántas conversiones más conseguimos llamando solo al 20% o 30% de clientes con mayor probabilidad predicha frente a llamar al azar.


In [ ]:
# Calculamos la ganancia acumulada para el mejor modelo (Random Forest)
rf_model = modelos['Random Forest']
probs = rf_model.predict_proba(X_test)[:, 1]

df_lift = pd.DataFrame({'real': y_test.values, 'prob': probs}).sort_values(by='prob', ascending=False)
df_lift['aciertos_acumulados'] = df_lift['real'].cumsum()
df_lift['pct_clientes'] = np.arange(1, len(df_lift) + 1) / len(df_lift) * 100
df_lift['pct_aciertos'] = df_lift['aciertos_acumulados'] / df_lift['real'].sum() * 100

plt.figure(figsize=(9, 5))
plt.plot(df_lift['pct_clientes'], df_lift['pct_aciertos'], color='#8e44ad', lw=2.5, label='Modelo Random Forest')
plt.plot([0, 100], [0, 100], 'k--', label='Llamadas Aleatorias (Baseline)')

plt.title("Curva de Ganancia Acumulada para Campaña de Telemarketing", fontsize=13, fontweight='bold')
plt.xlabel("% de Clientes Contactados (Ordenados por Score)")
plt.ylabel("% de Conversiones Totales Capturadas")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# KPI de Negocio: Contactando al top 30%
aciertos_top30 = df_lift[df_lift['pct_clientes'] <= 30]['pct_aciertos'].iloc[-1]
print(f"Impacto para Call Center:")
print(f"--> Contactando solo al 30% de clientes prioritarios capturamos el {aciertos_top30:.1f}% de todas las ventas potenciales.")


## 6. Conclusiones y Retorno de Inversión (ROI)

1. **Ahorro de Costes Operativos**: Priorizar el call center con este modelo permite reducir en un 70% las llamadas innecesarias a clientes que no están interesados, manteniendo más del 60% del volumen total de ventas.
2. **Interpretabilidad + Rendimiento**: La Regresión Logística y el Random Forest ofrecen un balance perfecto entre explicabilidad técnica para compliance bancario y potencia de discriminación comercial.
